# Pi Pulse Number Sweep Hardware Test

This notebook tests `qickdawg.pi_pulse_number_sweep.PiPulseNumberSweep` on the testing hardware at `128.95.31.224`.

Use it like `fine_time_counting_demo.ipynb`: connect to the QICK server, build an `NVConfiguration`, instantiate the pulse program, then call `prog.acquire(...)`.

The first oscilloscope test is deliberately countable (`N = 5, 6, 7, 8`) so the pulse spacing and number of pi pulses are easy to verify before running long trains.

## Oscilloscope Setup

- Connect CH1 to the MW output path with proper attenuation and 50 ohm termination.
- Optionally connect CH2 to the laser/PMOD marker to locate experiment boundaries.
- Turn off unused scope channels while debugging the MW waveform so the scope can use its highest sample rate.
- Use DC coupling, full bandwidth, and the highest available sample rate. At 100 MHz, 500 MSa/s is usable; 1-5 GSa/s is better.
- Trigger on CH1 rising edge at a small threshold above noise, or trigger on the marker if CH1 is unstable.
- With reps as the outer loop and the `n` sweep as the inner loop, each repetition should emit the configured `N` values in order.
- The default settings below use a scope-friendly 100 MHz carrier and a 50 ns pi pulse. These are for waveform validation, not NV-resonant contrast.
- Keep `mw_gain` low for the first pass, then increase only if the scope signal is too small.

In [ ]:
%load_ext autoreload
%autoreload 2

import inspect
from copy import copy

import numpy as np
import qickdawg as qd
from qickdawg.pi_pulse_number_sweep import PiPulseNumberSweep

In [ ]:
import Pyro4

Pyro4.config.SERIALIZER = "pickle"
Pyro4.config.SERIALIZERS_ACCEPTED = {"pickle"}
Pyro4.config.PICKLE_PROTOCOL_VERSION = 4

ns = Pyro4.locateNS(host="128.95.31.224", port=8888)

for name, uri in ns.list().items():
    print(name, uri)

In [ ]:
qd.start_client("128.95.31.224")

## Server Package Check

Run this before touching the hardware. The printed paths should point to the checkout/package that contains `PiPulseNumberSweep`, not an older installed copy.

In [ ]:
print("qickdawg package:", qd.__file__)
print("PiPulseNumberSweep source:", inspect.getsourcefile(PiPulseNumberSweep))

## Baseline Configuration

These values match the fine-time counting demo style. Adjust channels, frequency, gain, and thresholds for the actual setup before running the hardware cells.

In [ ]:
default_config = qd.NVConfiguration()

# Hardware channels
default_config.mw_channel = 2
default_config.adc_channel = 3
default_config.laser_gate_pmod = 0

# Photon counting parameters
default_config.edge_counting = True
default_config.high_threshold = 8000
default_config.low_threshold = 500

# MW pulse parameters
# Scope-debug setting: PiPulseNumberSweep uses pi duration = 2 * mw_pi2_ftns.
# A 25 ns pi/2 setting therefore produces a 50 ns pi pulse.
default_config.mw_pi2_ftns = 25.0
default_config.mw_nqz = 1
default_config.mw_fMHz = 100.0
default_config.mw_gain = 1000

# Timing delays
default_config.mw_to_laser_delay_tns = 555
default_config.relax_delay_tus = 2

# Readout timing
default_config.laser_on_tus = 6
default_config.readout_reference_start_tus = 5
default_config.readout_integration_tns = 633
default_config.laser_readout_offset_tus = 1.159

# Experiment control
default_config.reps = 500
default_config.pre_init = True
default_config.get_reference = False

default_config.validate_ft_samps_per_clk()

In [ ]:
def make_pi_count_config(
    n_start=0,
    n_end=4,
    n_delta=1,
    tau_ftns=500,
    tau_ftsamp=None,
    reps=500,
    mw_gain=1000,
    mw_fMHz=None,
):
    config = copy(default_config)
    config.reps = int(reps)
    config.mw_gain = int(mw_gain)
    if mw_fMHz is not None:
        config.mw_fMHz = float(mw_fMHz)

    if tau_ftsamp is None:
        config.tau_ftns = tau_ftns
    else:
        config.tau_ftsamp = int(tau_ftsamp)

    config.add_unitless_linear_sweep("n", int(n_start), int(n_end), delta=int(n_delta))
    return config


def print_expected_scope_timing(config):
    pi_width_ns = 2 * config.mw_pi2_ftns
    active_gap_ns = config.tau_ftns
    leading_edge_spacing_ns = pi_width_ns + active_gap_ns
    carrier_period_ns = 1000 / config.mw_fMHz
    cycles_per_pi = pi_width_ns / carrier_period_ns
    n_values = np.arange(config.n_start, config.n_end + config.n_delta, config.n_delta)
    train_duration_ns = np.where(
        n_values > 0,
        (n_values - 1) * leading_edge_spacing_ns + pi_width_ns,
        0,
    )

    print(f"n sweep: {config.n_start} to {config.n_end} by {config.n_delta}")
    print(f"carrier: {config.mw_fMHz:.6g} MHz, period: {carrier_period_ns:.6g} ns")
    print(f"tau: {config.tau_ftsamp} ftsamp = {config.tau_ftns:.6g} ns")
    print(f"pi width: {pi_width_ns:.6g} ns")
    print(f"RF cycles per pi pulse: {cycles_per_pi:.3g}")
    print(f"expected active end-to-start gap: {active_gap_ns:.6g} ns")
    print(f"expected adjacent leading-edge spacing: {leading_edge_spacing_ns:.6g} ns")
    print(f"expected train duration range: {train_duration_ns.min() / 1000:.6g} to {train_duration_ns.max() / 1000:.6g} us")


def build_program_and_check_axis(config, expected_points):
    prog = PiPulseNumberSweep(config)
    sweep_pts = np.asarray(prog.qick_sweeps[0].get_sweep_pts(), dtype=int)
    expected_points = np.asarray(expected_points, dtype=int)
    print("sweep points:", sweep_pts)
    np.testing.assert_array_equal(sweep_pts, expected_points)
    return prog, sweep_pts

## First Scope Sweep: `N = 5, 6, 7, 8`

Expected software result: the sweep axis is exactly `[5, 6, 7, 8]`.

Expected scope result:
- `N=5`: exactly five MW pulses.
- `N=6`: exactly six MW pulses.
- `N=7`: exactly seven MW pulses.
- `N=8`: exactly eight MW pulses.
- Adjacent active end-to-start gap is `tau_ftns`.
- Adjacent leading-edge spacing is `2 * mw_pi2_ftns + tau_ftns`.
- Suggested scope view: `500 ns/div` or `1 us/div`, max sample rate/memory, CH1 trigger rising just above noise.

In [ ]:
boundary_config = make_pi_count_config(n_start=5, n_end=8, n_delta=1, tau_ftns=500)
print_expected_scope_timing(boundary_config)

boundary_prog, boundary_sweep_pts = build_program_and_check_axis(
    boundary_config,
    expected_points=[5, 6, 7, 8],
)

In [ ]:
# Run this cell while watching the oscilloscope.
boundary_raw_data = boundary_prog.acquire(raw_data=True, progress=True)

## Long-Train Scope Sweep: `N = 60, 61, 62, 63, 64`

Run this only after the `N=5..8` sweep shows the right spacing. At the default 50 ns pi pulse and 500 ns tau, the total train should be about 32.5 to 34.7 us long.

Suggested scope view: `5 us/div` or `10 us/div`, max sample rate/memory. If the sample rate drops below about `500 MSa/s`, increase memory depth or temporarily use an even lower carrier such as `50 MHz`.

In [ ]:
long_train_config = make_pi_count_config(n_start=60, n_end=64, n_delta=1, tau_ftns=500)
print_expected_scope_timing(long_train_config)

long_train_prog, long_train_sweep_pts = build_program_and_check_axis(
    long_train_config,
    expected_points=[60, 61, 62, 63, 64],
)

In [ ]:
# Run this cell while watching the oscilloscope at 5 us/div or 10 us/div.
long_train_raw_data = long_train_prog.acquire(raw_data=True, progress=True)

## Fine Timing Step Check

This reruns a smaller `N=2,3` sweep at three closely spaced `tau_ftsamp` values.

Acceptance criteria:
- `base + 1` shifts adjacent leading-edge spacing by one DAC sample.
- `base + 16` shifts adjacent leading-edge spacing by one tProc clock when `samps_per_clk = 16`.
- Pulse count still matches `N` for both sweep points.

In [ ]:
base_tau_ftsamp = boundary_config.tau_ftsamp
tau_variants = [base_tau_ftsamp, base_tau_ftsamp + 1, base_tau_ftsamp + 16]

fine_timing_programs = []
for tau in tau_variants:
    config = make_pi_count_config(n_start=2, n_end=3, n_delta=1, tau_ftsamp=tau)
    print("\n--- tau variant ---")
    print_expected_scope_timing(config)
    prog, sweep_pts = build_program_and_check_axis(config, expected_points=[2, 3])
    fine_timing_programs.append((tau, config, prog))

In [ ]:
# Run these one at a time while watching the oscilloscope.
# Capture each tau variant separately.
fine_timing_raw_data = {}


def run_fine_timing_variant(index):
    tau, config, prog = fine_timing_programs[index]
    print(f"Running tau_ftsamp={tau}, tau_ftns={config.tau_ftns:.6g} ns")
    fine_timing_raw_data[int(tau)] = prog.acquire(raw_data=True, progress=True)
    return fine_timing_raw_data[int(tau)]


# Uncomment one line at a time, capture the scope trace, then run the next.
# run_fine_timing_variant(0)
# run_fine_timing_variant(1)
# run_fine_timing_variant(2)

## Phase Alternation Check

For `N=4`, pulses should alternate `X, Y, X, Y`. If the scope can show RF phase at the configured 100 MHz debug carrier, compare adjacent pulse phase. If direct phase is still hard to see, record phase as not directly scope-verified.

In [ ]:
# Use N=3..4 here and inspect the N=4 train. A single-point N=4 sweep is
# intentionally not used because PiPulseNumberSweep requires nsweep_points >= 2.
phase_config = make_pi_count_config(n_start=3, n_end=4, n_delta=1, tau_ftns=500)
print_expected_scope_timing(phase_config)
phase_prog, phase_sweep_pts = build_program_and_check_axis(phase_config, expected_points=[3, 4])

In [ ]:
# Run this while checking that the N=4 train follows X, Y, X, Y.
phase_raw_data = phase_prog.acquire(raw_data=True, progress=True)